# 6. Mixed-integer linear programme

Lecture: *Week 3 - Linear Programming 2*, slides 25–26.

*Original script: `5 Source Files/Python/milp_pyomo.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *

In [ ]:
# Create a concrete model
model = ConcreteModel()

In [ ]:
# Decision variables
model.x1 = Var(domain=NonNegativeIntegers, bounds=(0, float('inf')))
model.x2 = Var(domain=NonNegativeReals, bounds=(0, float('inf')))

In [ ]:
# Objective function (maximize 2x1 + 3.5x2)
f = [2, 3.5]
model.obj = Objective(expr=f[0]*model.x1 + f[1]*model.x2, sense=maximize)

In [ ]:
# Constraints
model.c1 = Constraint(expr=3*model.x1 + 2*model.x2 <= 14)
model.c2 = Constraint(expr=2*model.x1 + 3*model.x2 <= 12.5)
model.c3 = Constraint(expr=-model.x1 + model.x2 <= 1)

In [ ]:
# Print model before solving
print("Model formulation:")
model.pprint()

In [ ]:
# Solve with HiGHS
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)

In [ ]:
# Check results
if (results.solver.status == SolverStatus.ok) and \
   (results.solver.termination_condition == TerminationCondition.optimal):
    print("\nOptimal solution found:")
    model.display()
else:
    print("\nSolver did not find an optimal solution. Status:", results.solver.status)